# Pairwise KS test

Two-sample Kolmogorov-Smirnov test between model pairs. Run from repo root after CHRR sampling.

In [ ]:
import cobra
import pandas as pd
from scipy.stats import ks_2samp

MODEL_DIR = "model/contextualized_models"
RANGE_MIN = 0.001

# Seven pairwise comparisons
PAIRS = [
    ("egg_c", "L1_c"),
    ("L1_c", "L2_c"),
    ("L2_c", "L3_c"),
    ("L3_c", "L4mf_union_c"),
    ("L4f_c", "Af_c"),
    ("L4m_c", "Am_c"),
    ("Af_c", "Am_c"),
]

print(f"Comparisons: {len(PAIRS)}")
print(f"Range threshold: {RANGE_MIN}")

In [ ]:
# Load flux samples
STAGES = sorted({s for pair in PAIRS for s in pair})
samples = {s: pd.read_csv(f"{s}_samples.csv") for s in STAGES}
ranges = {s: df.max() - df.min() for s, df in samples.items()}

print(f"Stages: {', '.join(STAGES)}")
for stage in STAGES:
    print(f"  {stage}: {len(samples[stage])} samples")

In [ ]:
# Load subsystem annotations
subsystem = {}
for stage in STAGES:
    model = cobra.io.read_sbml_model(f"{MODEL_DIR}/{stage}.xml")
    for rxn in model.reactions:
        subsystem.setdefault(rxn.id, rxn.subsystem)

print(f"Subsystems loaded: {len(subsystem)}")

In [ ]:
# Benjamini-Hochberg FDR correction
def bh_adjust(pvals):
    ranks = pvals.rank(method="first")
    n = len(pvals)
    q = pvals * n / ranks
    running_min = 1.0
    q_adj = pd.Series(index=pvals.index, dtype=float)
    for idx in pvals.sort_values(ascending=False).index:
        running_min = min(running_min, q[idx])
        q_adj[idx] = running_min
    return q_adj.clip(upper=1.0)

print("BH correction function defined")

In [ ]:
# Run pairwise tests
for name1, name2 in PAIRS:
    df1, df2 = samples[name1], samples[name2]
    shared = sorted(set(df1.columns) & set(df2.columns))
    
    # Filter: unsupported + range >= RANGE_MIN
    shared = [r for r in shared if subsystem.get(r, "") != "Unsupported"]
    r1, r2 = ranges[name1], ranges[name2]
    shared = [r for r in shared if r1[r] >= RANGE_MIN and r2[r] >= RANGE_MIN]
    
    # KS test
    rows = []
    for rxn in shared:
        d, p = ks_2samp(df1[rxn], df2[rxn])
        med1, med2 = df1[rxn].median(), df2[rxn].median()
        pooled_range = max(df1[rxn].max(), df2[rxn].max()) - min(df1[rxn].min(), df2[rxn].min())
        rows.append((rxn, subsystem.get(rxn, ""), d, p, med1, med2, med2 - med1, pooled_range))
    
    result = pd.DataFrame(rows, columns=[
        "rxnID", "subsystem", "D", "pvalue", f"median_{name1}", f"median_{name2}",
        "delta_median", "pooled_range",
    ])
    
    result["qvalue"] = bh_adjust(result["pvalue"])
    result["relative_delta"] = result["delta_median"].abs() / result["pooled_range"]
    result["rank_D"] = result["D"].rank(ascending=False, method="min").astype(int)
    result["rank_relative_delta"] = result["relative_delta"].rank(ascending=False, method="min").astype(int)
    result["combined_score"] = result["rank_D"] + result["rank_relative_delta"]
    result = result.sort_values("combined_score").reset_index(drop=True)
    
    result.to_csv(f"pairwise_ks_{name1}_vs_{name2}.csv", index=False)
    n_sig = (result["qvalue"] < 0.05).sum()
    print(f"{name1} vs {name2}: {len(shared)} reactions, {n_sig} significant (q<0.05)")

In [ ]:
print("\n✓ Saved 7 pairwise KS result files")